# Magnetohydrodynamic Flow

This notebook compares MI-DL and SI-DL dimensionless coordinates for the MHD dataset. It reruns both searches, evaluates the coordinates, and retains a GPR fit comparison and a summary table.


## 1. Imports, data, and settings

The source pickle is stored locally in `data/mhd.pkl`. Search results are written to `csv/`, while only `fit.png` and `summary.png` are retained in `figures/`.


In [ ]:
from __future__ import annotations

import os

import pickle

import sys

import warnings

from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-codex-cache")

os.environ.setdefault("MPLBACKEND", "Agg")

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.linalg import null_space

from scipy.optimize import differential_evolution

OUTPUT_DIR = Path.cwd()

ROOT = OUTPUT_DIR.parent

for module_dir in [ROOT / "Compare" / "MI-DL", ROOT / "SI-DL-main"]:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import midl

import SI_DL

RANDOM_STATE = 43

FIG_DIR = OUTPUT_DIR / "figures"

CSV_DIR = OUTPUT_DIR / "csv"

SOURCE_PICKLE = OUTPUT_DIR / "data" / "mhd.pkl"

VARIABLE_LABELS = ["l", "mu", "rho", "eta", "dp_dx", "B"]

MI_K_NEIGHBORS = 6

MI_DE_MAXITER = 120

MI_RESTART_SEEDS = [42]

SI_BANDWIDTH = 0.5

SI_MAXITER = 800

SI_POPSIZE = 30

D_IN = np.array(
    [
        [1, -1, -3, 3, -2, 0],
        [0, -1, 0, -3, -2, -2],
        [0, 1, 1, 1, 1, 1],
        [0, 0, 0, -2, 0, -1],
    ],
    dtype=float,
)


## 2. Load data and construct candidate coordinates

Read the physical inputs and response, normalize exponent vectors, and compute dimensionless coordinates and common scores.


In [ ]:
def load_mhd_data() -> tuple[np.ndarray, np.ndarray]:
    with SOURCE_PICKLE.open("rb") as handle:
        data = pickle.load(handle)

    X = np.asarray(data["X"], dtype=float)
    Y = np.asarray(data["Y"], dtype=float).reshape(-1)

    return X, Y

def normalize_exponents(exponents: np.ndarray) -> np.ndarray:
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))

    if scale <= 1e-12:
        return row

    out = row / scale
    first = np.flatnonzero(np.abs(out) > 1e-10)

    if first.size and out[first[0]] < 0.0:
        out = -out

    return out

def pi_from_exponents(X: np.ndarray, exponents: np.ndarray) -> np.ndarray:
    return np.exp(
        np.log(np.asarray(X, dtype=float))
        @ np.asarray(exponents, dtype=float).reshape(-1)
    )

def formula_from_exponents(exponents: np.ndarray, decimals: int = 4) -> str:
    terms = []

    for label, value in zip(VARIABLE_LABELS, np.asarray(exponents).reshape(-1)):
        value = float(value)

        if abs(value) < 5e-5:
            continue

        terms.append(f"{label}^{value:.{decimals}f}")

    return " * ".join(terms)

def information_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    info = midl.information_lower_bound(
        np.asarray(feature, dtype=float).reshape(-1, 1),
        Y,
        k_neighbors=MI_K_NEIGHBORS,
        random_state=RANDOM_STATE,
    )

    return {
        "mutual_information": float(info["mutual_information"]),
        "epsilon_lb_normalized": float(info["epsilon_lb"] / np.var(Y, ddof=0)),
    }


## 3. MI-DL and SI-DL searches

Search the dimensional null space using the information and covariance objectives.


In [ ]:
def run_midl_search(X: np.ndarray, Y: np.ndarray) -> dict:
    basis, _ = midl.calc_basis(D_IN)
    log_pi_basis = np.log(X) @ basis

    rows = []
    best = None

    for seed in MI_RESTART_SEEDS:
        model = midl.MIDL(
            k_neighbors=MI_K_NEIGHBORS,
            de_maxiter=MI_DE_MAXITER,
            random_state=seed,
        )

        w, _search_mi, _ = model._optimize_direction_in_subspace(
            log_pi_basis,
            Y,
            np.eye(basis.shape[1]),
        )

        exponents = normalize_exponents(np.asarray(basis @ w).reshape(-1))
        pi = pi_from_exponents(X, exponents)
        mi = information_metrics(pi, Y)

        row = {
            "seed": seed,
            "mi_raw_pi": mi["mutual_information"],
            "epsilon_lb_raw_pi_normalized": mi["epsilon_lb_normalized"],
            "formula": formula_from_exponents(exponents),
            "exponents": exponents,
        }

        rows.append(row)

        if best is None or row["mi_raw_pi"] > best["mi_raw_pi"]:
            best = row

    audit = pd.DataFrame(
        [{k: v for k, v in row.items() if k != "exponents"} for row in rows]
    )
    audit = audit.sort_values("mi_raw_pi", ascending=False)

    return {
        "exponents": best["exponents"],
        "best_seed": int(best["seed"]),
        "restart_audit": audit,
    }

def run_sidl_search(X: np.ndarray, Y: np.ndarray) -> dict:
    basis = null_space(D_IN).astype(float)

    def params_to_exponents(params: np.ndarray) -> np.ndarray:
        return normalize_exponents(
            basis @ np.asarray(params, dtype=float).reshape(-1)
        )

    def objective(params: np.ndarray) -> float:
        try:
            pi = pi_from_exponents(X, params_to_exponents(params))
            score = SI_DL.explained_variance_score(
                pi,
                Y,
                bandwidth=SI_BANDWIDTH,
            )["S_cov"]

        except Exception:
            return 1e6

        if not np.isfinite(score):
            return 1e6

        return -float(score)

    result = differential_evolution(
        objective,
        bounds=[(-2.0, 2.0)] * basis.shape[1],
        maxiter=SI_MAXITER,
        popsize=SI_POPSIZE,
        seed=RANDOM_STATE,
        polish=True,
        updating="immediate",
        workers=1,
    )

    return {
        "exponents": params_to_exponents(result.x),
        "optimizer_result": result,
    }

def score_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    sidl_score = SI_DL.explained_variance_score(
        feature,
        Y,
        bandwidth=SI_BANDWIDTH,
    )

    mi = information_metrics(feature, Y)

    return {
        **mi,
        "S_cov": float(sidl_score["S_cov"]),
        "sidl_error": float(1.0 - sidl_score["S_cov"]),
        "sidl_bandwidth": float(sidl_score["bandwidth"]),
        "sidl_n_retained": int(sidl_score["n_retained"]),
    }


## 4. Summary table

Render the method comparison as one of the two retained images.


In [ ]:
def plot_summary_table(summary: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(14.8, 4.8), dpi=220)

    ax.axis("off")
    fig.patch.set_facecolor("white")

    ax.text(
        0.5,
        0.96,
        "MHD comparison, k=6",
        ha="center",
        va="top",
        fontsize=18,
        weight="bold",
    )

    rows = []

    for _, row in summary.iterrows():
        rows.append(
            [
                row["method"],
                row["formula"],
                f"{row['mutual_information']:.4f}",
                f"{row['epsilon_lb_normalized']:.5f}",
                f"{row['S_cov']:.4f}",
                f"{row['sidl_error']:.5f}",
                f"{row['sidl_n_retained']}",
            ]
        )

    table = ax.table(
        cellText=rows,
        colLabels=[
            "Method",
            "Found pi",
            "MI k=6",
            "epsilon_LB/Var",
            "S_cov",
            "1 - S_cov",
            "n retained",
        ],
        cellLoc="center",
        colLoc="center",
        colWidths=[0.12, 0.45, 0.09, 0.10, 0.08, 0.08, 0.08],
        bbox=[0.02, 0.10, 0.96, 0.76],
    )

    table.auto_set_font_size(False)

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#3f3f46")
        cell.set_linewidth(0.85)
        cell.PAD = 0.03

        text = cell.get_text()

        if r == 0:
            cell.set_facecolor("#1f2937")
            text.set_color("white")
            text.set_weight("bold")
            text.set_fontsize(10.0)
        else:
            cell.set_facecolor("#f8fafc" if r % 2 == 1 else "#ffffff")
            text.set_fontsize(8.0 if c == 1 else 9.8)

            if c == 1:
                text.set_ha("left")

            if c == 0:
                text.set_weight("bold")

    fig.savefig(
        FIG_DIR / "summary.png",
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


## 5. Run the dimensionless-coordinate comparison

Execute both searches and save the summary, restart audit, coordinates, and exponents.


In [ ]:
def run_comparison() -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    X, Y = load_mhd_data()

    midl_result = run_midl_search(X, Y)
    sidl_result = run_sidl_search(X, Y)

    coordinates = {
        "MI-DL": {"exponents": midl_result["exponents"]},
        "SI-DL": {"exponents": sidl_result["exponents"]},
    }

    for values in coordinates.values():
        values["pi"] = pi_from_exponents(X, values["exponents"])

    rows = []

    for method, values in coordinates.items():
        common = score_metrics(values["pi"], Y)

        rows.append(
            {
                "method": method,
                "formula": formula_from_exponents(values["exponents"]),
                "exponents": values["exponents"],
                "n_samples": int(Y.size),
                "feature_space": "raw_pi",
                **common,
                "best_seed": midl_result["best_seed"] if method == "MI-DL" else np.nan,
            }
        )

    summary = pd.DataFrame(rows)

    summary["rank_by_MI"] = summary["mutual_information"].rank(
        ascending=False,
        method="min",
    ).astype(int)

    summary["rank_by_S_cov"] = summary["S_cov"].rank(
        ascending=False,
        method="min",
    ).astype(int)

    summary_csv = summary.copy()

    for j, label in enumerate(VARIABLE_LABELS):
        summary_csv[f"pi1_exp_{label}"] = summary_csv["exponents"].map(
            lambda arr, jj=j: arr[jj]
        )

    summary_csv.drop(columns=["exponents"]).to_csv(
        CSV_DIR / "summary.csv",
        index=False,
    )

    midl_result["restart_audit"].to_csv(
        CSV_DIR / "midl_restarts.csv",
        index=False,
    )

    coord_out = pd.DataFrame(X, columns=VARIABLE_LABELS)
    coord_out["Y"] = Y

    for method, values in coordinates.items():
        coord_out[f"{method}_pi1"] = values["pi"]

    coord_out.to_csv(
        CSV_DIR / "coordinates.csv",
        index=False,
    )

    exponent_rows = []

    for method, values in coordinates.items():
        for label, exponent in zip(VARIABLE_LABELS, values["exponents"]):
            exponent_rows.append(
                {
                    "method": method,
                    "variable": label,
                    "normalized_exponent": float(exponent),
                }
            )

    pd.DataFrame(exponent_rows).to_csv(
        CSV_DIR / "exponents.csv",
        index=False,
    )

    plot_summary_table(summary)

    return {"summary": summary, "coordinates": coordinates, "Y": Y}


In [ ]:
outputs = run_comparison()
outputs["summary"]


## 6. Refit GPR models and create the fit comparison

Use the newly generated MI-DL and SI-DL coordinates with a fixed one-dimensional GPR specification.


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Y = outputs["Y"]
coordinates = outputs["coordinates"]
train_idx, test_idx = train_test_split(np.arange(Y.size), test_size=0.25, random_state=42)
methods = (("MI-DL", 12.0), ("SI-DL", 1.5))
fits = {}; gpr_rows = []
for method, length_scale in methods:
    feature = coordinates[method]["pi"].reshape(-1, 1)
    model = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=ConstantKernel(1.0) * RBF(length_scale) + WhiteKernel(1e-6),
            alpha=1e-10,
            normalize_y=True,
            optimizer=None,
            n_restarts_optimizer=0,
            random_state=42,
        ),
    )
    model.fit(feature[train_idx], Y[train_idx])
    prediction = model.predict(feature[test_idx])
    mse_raw = float(mean_squared_error(Y[test_idx], prediction))
    mse_normalized = mse_raw / float(np.var(Y[train_idx], ddof=0))
    fits[method] = model
    gpr_rows.append({"method":method,"gpr_mse_raw":mse_raw,"gpr_mse_normalized":mse_normalized})
gpr_summary = pd.DataFrame(gpr_rows)
gpr_summary.to_csv(CSV_DIR / "gpr_summary.csv", index=False)

plt.rcParams.update({"font.family":"STIXGeneral","mathtext.fontset":"stix","axes.titlesize":22,"axes.labelsize":21,"xtick.labelsize":17,"ytick.labelsize":17,"legend.fontsize":15})
fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.7), sharey=True, dpi=260)
for ax, (method, _) in zip(axes, methods):
    feature = coordinates[method]["pi"].reshape(-1, 1)
    line_x = np.linspace(float(feature.min()), float(feature.max()), 320).reshape(-1, 1)
    line_y = fits[method].predict(line_x)
    mse = float(gpr_summary.loc[gpr_summary["method"].eq(method),"gpr_mse_normalized"].iloc[0])
    ax.scatter(feature[:,0],Y,s=42,color="#1f77b4",alpha=0.58,edgecolors="white",linewidths=0.45,label="data")
    ax.plot(line_x[:,0],line_y,color="#d62728",linewidth=2.9,label="GPR mean")
    ax.text(0.97,0.06,f"GPR norm. MSE = {mse:.4f}",transform=ax.transAxes,ha="right",va="bottom",fontsize=15,bbox={"boxstyle":"round,pad=0.24","facecolor":"white","edgecolor":"#d1d5db","alpha":0.92})
    ax.set_xlabel(rf"{method} $\Pi_1$")
    ax.set_title(method)
    ax.grid(True,alpha=0.22)
    ax.legend(frameon=True,loc="upper left")
axes[0].set_ylabel(r"$u\rho l/\mu$")
fig.suptitle("MHD",fontsize=25,y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fit.png",bbox_inches="tight",facecolor="white")
plt.close(fig)
gpr_summary
